# Lab 1: State & Nodes
In LangGraph, the central hub of information is the **State**. It represents the shared memory/context of the graph. Nodes are Python functions that receive the current state and return state updates (as a dictionary).

Let's build a simple sequential graph that:
1. Receives an input text.
2. Converts it to uppercase in Node 1.
3. Appends exclamation marks in Node 2.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Verify API keys
print("OpenAI API Key set:", "OPENAI_API_KEY" in os.environ)
print("LangSmith tracing set:", os.environ.get("LANGCHAIN_TRACING_V2"))

OpenAI API Key set: True
LangSmith tracing set: true


### Define State, Nodes, and compile the Graph

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. Define the State structure using TypedDict
class SimpleState(TypedDict):
    input_text: str
    processed_text: str
    counter: int

# 2. Define the Nodes (Python functions)
def uppercase_node(state: SimpleState):
    print("--- Executing uppercase_node ---")
    return {
        "processed_text": state["input_text"].upper(),
        "counter": state.get("counter", 0) + 1
    }

def punctuation_node(state: SimpleState):
    print("--- Executing punctuation_node ---")
    return {
        "processed_text": state["processed_text"] + "!!!",
        "counter": state["counter"] + 1
    }

# 3. Build the Graph
builder = StateGraph(SimpleState)

# Add nodes to the graph
builder.add_node("uppercase", uppercase_node)
builder.add_node("punctuation", punctuation_node)

# Add edges to connect nodes
builder.add_edge(START, "uppercase")
builder.add_edge("uppercase", "punctuation")
builder.add_edge("punctuation", END)

# 4. Compile the graph into an executable runnable
graph = builder.compile()

# Draw/Visualize the graph
try:
    graph.get_graph().print_ascii()
except Exception as e:
    print("Unable to display ASCII graph:", e)

 +-----------+   
 | __start__ |   
 +-----------+   
        *        
        *        
        *        
 +-----------+   
 | uppercase |   
 +-----------+   
        *        
        *        
        *        
+-------------+  
| punctuation |  
+-------------+  
        *        
        *        
        *        
  +---------+    
  | __end__ |    
  +---------+    


### Run the Graph

In [3]:
initial_state = {"input_text": "hello langgraph", "counter": 0}
final_state = graph.invoke(initial_state)

print("\n--- Final Result State ---")
print("Processed Text:", final_state["processed_text"])
print("Counter:", final_state["counter"])

--- Executing uppercase_node ---
--- Executing punctuation_node ---

--- Final Result State ---
Processed Text: HELLO LANGGRAPH!!!
Counter: 2
